# INF01090 - Ciência de Dados

# Models Lab

This lab is about **understanding how models behave** before studying specific regression and classification algorithms in detail.

## Goals

By the end of this lab, you should be able to:

- see a model as a function from features to predictions
- explore how changing parameters changes predictions
- understand the role of train/test separation
- recognize underfitting and overfitting
- understand decision boundaries visually
- see how feature engineering can make a problem easier
- explain why representation matters in modeling

## Lab format

Work in groups of up to 4 students.

This lab is **not** about choosing the best algorithm yet.
It is about building intuition.


In [3]:
import numpy as np
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()
np.random.seed(42)


# Part A — Regression Intuition


## 1. A simple regression dataset

The points below follow a noisy linear trend generated from a hidden line. 


In [4]:
x = [0.0000, 0.4167, 0.8333, 1.2500, 1.6667, 2.0833, 2.5000, 2.9167, 3.3333, 3.7500, 4.1667, 4.5833, 5.0000, 5.4167, 5.8333, 6.2500, 6.6667, 7.0833, 7.5000, 7.9167, 8.3333, 8.7500, 9.1667, 9.5833, 10.0000]

y = [5.5982, 8.2860, 5.7088, 6.6459, 11.2097, 10.5233, 8.9866, 11.9478, 10.8730, 11.8054, 14.1579, 10.7846, 12.0984, 15.3608, 15.3968, 18.9881, 17.4807, 17.4092, 24.1023, 21.6565, 23.1802, 21.1328, 23.8307, 26.0784, 24.4917]

x = np.array(x)
y = np.array(y)

df_reg = pd.DataFrame({"x": x, "y": y})


alt.Chart(df_reg).mark_circle(size=80).encode(
    x=alt.X("x:Q", title="Feature"),
    y=alt.Y("y:Q", title="Target"),
    tooltip=["x:Q", "y:Q"]
).properties(
    width=600,
    height=350,
    title="Regression Dataset"
)


alt.Chart(...)

## 2. Your first manual model

A simple linear model can be written as:

\[
$\hat{y} = a x + b$
\]

where:
- `a` controls the slope
- `b` controls the intercept


In [21]:
a = 2.0
b = 5.0

df_reg["y_pred"] = a * df_reg["x"] + b

points = alt.Chart(df_reg).mark_circle(size=80).encode(
    x="x:Q",
    y="y:Q"
)

line = alt.Chart(df_reg).mark_line(color="red", strokeWidth=3).encode(
    x="x:Q",
    y="y_pred:Q"
)

(points + line).properties(
    width=600,
    height=350,
    title="Manual Linear Model"
)


alt.LayerChart(...)

### Tasks
- Change `a`
- Change `b`
- Try to improve the fit visually
- What happens when you increase `a`?
- What happens when you increase `b`?


### **Answers Part 2**: Write your group answers here.

1. a (inclinação) deixa a reta mais íngreme, enquanto aumentar b (intercepto) desloca a reta inteira para cima no eixo Y.

## 3. Learning as parameter adjustment

In real machine learning, the model adjusts parameters automatically.

Here you are doing it **manually**:
- observe the data
- change the parameters
- improve the fit

Your goal is **not** to recover an exact formula from the code.
Your goal is to estimate a reasonable slope and intercept from the scatterplot.

In [27]:
# Try your own values here
# Start from a rough guess and refine it visually
a = 1.75
b = 5.75

df_reg["y_pred"] = a * df_reg["x"] + b

points = alt.Chart(df_reg).mark_circle(size=80).encode(
    x="x:Q",
    y="y:Q"
)

line = alt.Chart(df_reg).mark_line(color="green", strokeWidth=3).encode(
    x="x:Q",
    y="y_pred:Q"
)

(points + line).properties(
    width=600,
    height=350,
    title="Improved Manual Fit"
)


alt.LayerChart(...)

### Tasks
Try different values for `a` and `b` until the line seems to match the overall trend of the data. Return the values for  that you found.

### **Answers Part 3**: Write your group answers here.

1. a = 1.75, b = 5.75

## 4. Train vs Test

A model can look good on familiar data and still fail on new data.

We therefore separate data into:
- **Train**: used to fit the model
- **Test**: used only to check generalization


In [28]:
df_reg["set"] = ["Train" if i % 3 != 0 else "Test" for i in range(len(df_reg))]

alt.Chart(df_reg).mark_circle(size=90).encode(
    x=alt.X("x:Q", title="Feature"),
    y=alt.Y("y:Q", title="Target"),
    color=alt.Color("set:N", title="Dataset"),
    tooltip=["x:Q", "y:Q", "set:N"]
).properties(
    width=600,
    height=350,
    title="Train vs Test Split"
)


alt.Chart(...)

In [30]:
np.random.seed(42)

mask = np.random.rand(len(df_reg)) < 0.7  # 70% train

df_reg["set"] = np.where(mask, "Train", "Test")

alt.Chart(df_reg).mark_circle(size=90).encode(
    x=alt.X("x:Q", title="Feature"),
    y=alt.Y("y:Q", title="Target"),
    color=alt.Color("set:N", title="Dataset"),
    tooltip=["x:Q", "y:Q", "set:N"]
).properties(
    width=600,
    height=350,
    title="Train vs Test Split"
)

alt.Chart(...)

### Questions
- Why should the model only use the training data?
- Why is test data important?
- Why is it important that the test points follow the same distribution as the training points?

### **Answers Part 4**: Write your group answers here.

1. Para manter a integridade dos dados de testagem, o modelo não pode ver um exemplo que ele já viu antes pois ele saberá a resposta de antemão.
2. Para validar o aprendizado do modelo
3. Porque o modelo aprende com base me uma distribuição de amostras, e essa distribuição deve ser seguida nos dados de teste para não "confundir" o modelo.

## 5. Model complexity: underfitting and overfitting

A model can be too simple or too complex for the data.

- A **very simple** model may miss important patterns.
- A **very complex** model may follow noise instead of the true structure.

Use the slider below to change the polynomial degree and observe how the fitted curve changes.

### Task
Move the slider and identify:
- a degree that seems too simple
- a degree that seems too complex
- a degree that seems to capture the structure well

In [56]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
%pip install ipywidgets
import ipywidgets as widgets
from IPython.display import display, clear_output

np.random.seed(7)

x = np.linspace(0, 10, 30)
y_true = 0.5 * (x - 5)**2 + 3
y = y_true + np.random.normal(0, 1.8, len(x))

x_grid = np.linspace(x.min(), x.max(), 300)

out = widgets.Output()

def plot_polynomial_fit(degree):
    with out:
        clear_output(wait=True)

        model = make_pipeline(
            PolynomialFeatures(degree=degree, include_bias=False),
            LinearRegression()
        )
        model.fit(x.reshape(-1, 1), y)
        y_pred = model.predict(x_grid.reshape(-1, 1))

        plt.figure(figsize=(8, 4.5))
        plt.scatter(x, y)
        plt.plot(x_grid, y_pred, linewidth=2)
        plt.title(f"Polynomial fit (degree = {degree})")
        plt.xlabel("x")
        plt.ylabel("y")
        plt.show()

degree_slider = widgets.IntSlider(
    value=7,
    min=1,
    max=10,
    step=1,
    description="Degree:",
    continuous_update=False
)

widgets.interactive_output(plot_polynomial_fit, {"degree": degree_slider})
display(degree_slider, out)

plot_polynomial_fit(degree_slider.value)

IntSlider(value=7, continuous_update=False, description='Degree:', max=10, min=1)

Output()

### Questions
- What changes in the curve as the degree increases?
- At what point does the model start following small fluctuations in the data?
- Which degree gives a reasonable balance between simplicity and fit?

### **Answers Part 5**: Write your group answers here.

1. Sua curvatura e os pontos que ela abrange
2. A partir do grau 9
3. Depende do quanto de precisão queremos. Visando simplicidade, grau 2.

## 5. Underfitting and overfitting

Below there are two synthetic examples with three possible fits.


In [58]:
x_uo = np.linspace(0, 10, 25)
y_uo = np.sin(x_uo) + np.random.normal(0, 0.15, len(x_uo))
df_points = pd.DataFrame({"x": x_uo, "y": y_uo})

grid = np.linspace(0, 10, 300)
df_lines = pd.DataFrame({
    "x": np.concatenate([grid, grid, grid]),
    "y": np.concatenate([
        0.1 * grid + 0.3,
        np.sin(grid),
        np.sin(grid) + 0.2 * np.sin(6 * grid)
    ]),
    "fit": (["Underfit"] * len(grid)) + (["Good fit"] * len(grid)) + (["Overfit"] * len(grid))
})

points = alt.Chart(df_points).mark_circle(size=50, color="black").encode(
    x="x:Q", y="y:Q"
)

lines = alt.Chart(df_lines).mark_line(strokeWidth=3).encode(
    x="x:Q",
    y="y:Q",
    color=alt.Color("fit:N", title="Fit type")
)

(points + lines).properties(
    width=650,
    height=350,
    title="Underfitting, Good Fit, and Overfitting"
)


alt.LayerChart(...)

In [59]:
import numpy as np
import pandas as pd
import altair as alt

np.random.seed(8)

x_uo = np.linspace(0, 10, 25)
y_uo = 2.2 * np.exp(-((x_uo - 5.0)**2) / 3.5) + 0.15 * x_uo + np.random.normal(0, 0.12, len(x_uo))
df_points = pd.DataFrame({"x": x_uo, "y": y_uo})

grid = np.linspace(0, 10, 300)

underfit = 0.18 * grid + 0.45
goodfit = 2.2 * np.exp(-((grid - 5.0)**2) / 3.5) + 0.15 * grid
overfit = goodfit + 0.18 * np.sin(5.5 * grid)

df_lines = pd.DataFrame({
    "x": np.concatenate([grid, grid, grid]),
    "y": np.concatenate([underfit, goodfit, overfit]),
    "fit": (["Underfit"] * len(grid)) + (["Good fit"] * len(grid)) + (["Overfit"] * len(grid))
})

points = alt.Chart(df_points).mark_circle(size=50, color="black").encode(
    x=alt.X("x:Q", title="x"),
    y=alt.Y("y:Q", title="y")
)

lines = alt.Chart(df_lines).mark_line(strokeWidth=3).encode(
    x="x:Q",
    y="y:Q",
    color=alt.Color("fit:N", title="Fit type")
)

(points + lines).properties(
    width=650,
    height=350,
    title="Underfitting, Good Fit, and Overfitting"
)

alt.LayerChart(...)

### Questions

- What visual characteristics indicate that a model is too simple for the data?
- What visual characteristics indicate that a model is too complex?
- Which aspects of the data should a good model capture, and which should it ignore?

1. Linhas que "atravessam diretamente" os dados, com comportamento de linhas retas sem curvaturas e/ou linhas que não abrangem poucos ou nenhum do conjunto de dados.
2. Linhas extremamente acentuadas que abrangem todos/a maioria dos dados do conjunto de treino.
3. Estatisticamente, deve dar mais atenção a dados que estejam entre o primeiro e terceiro quartil da distribuição. Deve ignorar dados que estejam classificados como "outliers". 


# Part B — Classification Intuition


## 6. A simple classification dataset

Now each point belongs to one of two classes.


In [60]:
x1 = np.random.normal(0, 1.5, 80)
x2 = np.random.normal(0, 1.5, 80)
y_cls = (x1 + x2 > 0).astype(int)

df_clf = pd.DataFrame({"x1": x1, "x2": x2, "class": y_cls})

alt.Chart(df_clf).mark_circle(size=80).encode(
    x=alt.X("x1:Q", title="Feature 1"),
    y=alt.Y("x2:Q", title="Feature 2"),
    color=alt.Color("class:N", title="Class"),
    tooltip=["x1:Q", "x2:Q", "class:N"]
).properties(
    width=600,
    height=400,
    title="Classification Dataset"
)


alt.Chart(...)

### Questions
- Are the classes perfectly separable?
- Where would you draw a boundary?

### **Answers Part 6**: Write your group answers here.
1. Não perfeitamente, vemos que as existem amostrar similares de ambas as classes no intervalo entre x=-0.25 e y=0.1
2. Próximo do intervalo informado acima.

## 7. A manual decision boundary

A very simple classifier can be written as:

\[
$\text{predict class 1 if } x_1 + x_2 > t$
\]

where $t$ is a threshold.


In [63]:
threshold = 0.0
df_clf["pred"] = (df_clf["x1"] + df_clf["x2"] > threshold).astype(int)

boundary = pd.DataFrame({
    "x1": [-5, 5],
    "x2": [threshold + 5, threshold - 5]
})

points = alt.Chart(df_clf).mark_circle(size=80).encode(
    x="x1:Q",
    y="x2:Q",
    color=alt.Color("class:N", title="True class"),
    tooltip=["x1:Q", "x2:Q", "class:N", "pred:N"]
)

line = alt.Chart(boundary).mark_line(color="black", strokeWidth=3).encode(
    x="x1:Q",
    y="x2:Q"
)

(points + line).properties(
    width=600,
    height=400,
    title="Manual Decision Boundary"
)


alt.LayerChart(...)

### Tasks
- Change `threshold`
- Observe how the boundary moves
- Which threshold seems best?


### **Answers Part 7**: Write your group answers here.

Curiosamente, o threashold informado pelo exercício, o qual é 0 (zero).

## 8. Train vs Test in classification

We again separate the data into train and test.


In [64]:
df_clf["set"] = ["Train" if i % 3 != 0 else "Test" for i in range(len(df_clf))]

alt.Chart(df_clf).mark_point(size=90, filled=True).encode(
    x=alt.X("x1:Q", title="Feature 1"),
    y=alt.Y("x2:Q", title="Feature 2"),
    color=alt.Color("class:N", title="Class"),
    shape=alt.Shape(
        "set:N",
        scale=alt.Scale(domain=["Train", "Test"], range=["circle", "square"]),
        legend=alt.Legend(title="Dataset")
    ),
    tooltip=["x1:Q", "x2:Q", "class:N", "set:N"]
).properties(
    width=600,
    height=400,
    title="Classification: Train (circles) vs Test (squares)"
)


alt.Chart(...)

### Questions
- Why should the boundary be chosen using the training set?
- What could go wrong if the test set influenced the boundary?


### **Answers Part 8**: Write your group answers here.
1. Porque estamos explicitando para o modelo o que ele deve aprender
2. O modelo não saberá que direção ir e acabará aprendendo de modo irrealista.

# Part C — Feature Engineering and Representation


## 9. Transforming a feature

Sometimes a simple transformation can make a modeling problem easier.


In [65]:
df_clf["x2_sq"] = df_clf["x2"] ** 2

chart_before = alt.Chart(df_clf).mark_circle(size=70).encode(
    x=alt.X("x1:Q", title="Feature 1"),
    y=alt.Y("x2:Q", title="Feature 2"),
    color=alt.Color("class:N", legend=None)
).properties(
    width=280,
    height=280,
    title="Original feature space"
)

chart_after = alt.Chart(df_clf).mark_circle(size=70).encode(
    x=alt.X("x1:Q", title="Feature 1"),
    y=alt.Y("x2_sq:Q", title="Transformed feature x2²"),
    color=alt.Color("class:N", legend=None)
).properties(
    width=280,
    height=280,
    title="After feature transformation"
)

chart_before | chart_after


alt.HConcatChart(...)

### Questions
- Did the structure become easier to separate?
- Why can feature engineering improve models?


### **Answers Part 9**: Write your group answers here.
1. Não, as amostras de ambas as classes, após a mudança, estão compartilhando muito mais dados do que antes.
2. Porque alterando a visualização dos dados podemos fazer o modelo detectar insights não avaliados com os dados brutos.

# Part D — Group Challenge


## 10. Applications to datasets used in the visualization class

Describe regression or classification tasks that can be applied for the datasets you used in the visualization class (or other dataset). We are going to explore these tasks even further in the next lab class.



### **Answers Part 10**: Write your group answers here.
1. Explorando o dataset Produtividade e Distrações (https://www.kaggle.com/datasets/sehaj1104/student-productivity-and-digital-distraction-dataset): Nas tarefas de classificação, é possível prever em qual quartil de produtividade um aluno se encaixa com base em seus hábitos diários ou treinar um modelo para detectar se ele possui risco de distração (classificando-o como um outlier ou não a partir dos níveis de estresse e carga de estudo). Já para as tarefas de regressão, é possível estimar valores contínuos, como prever a nota final exata ou o score de produtividade do aluno a partir de suas horas de estudo, volume de distrações e frequência nas aulas, além de calcular seu nível de foco usando dados da rotina, como horas de sono e ingestão de café.